# Find topics in ODA 

In [13]:
import requests
import pandas as pd
from pathlib import Path
import requests

In [14]:
DATA_DIR = Path("../data/temp_data")

roll_calls_file = DATA_DIR / "parliament" / "roll_calls_resume.csv"

df_roll_calls = pd.read_csv(roll_calls_file)

df_cases = (
    df_roll_calls[
        [
            "sagid",
            "sag_nummer",
            "sag_titel",
            "sag_titelkort",
            "sag_resume",
            "dato",
        ]
    ]
    .drop_duplicates(subset="sagid")
    .reset_index(drop=True)
)

print("Unique parliamentary cases:", len(df_cases))
df_cases.head()

Unique parliamentary cases: 2128


,sagid,sag_nummer,sag_titel,sag_titelkort,sag_resume,dato
0,1449,L 200,Forslag til lov om ændring af virksomhedsskatt...,Om indgreb mod utilsigtet udnyttelse af virkso...,Loven ændrer virksomhedsskatteordningens regle...,2014-09-09 09:15:00
1,12715,L 4,Forslag til lov om ændring af lov om afgift af...,Om tilbagerulning af forsyningssikkerhedsafgif...,"Lovforslaget betyder, at afgiftsforhøjelserne ...",2014-10-31 10:00:00
2,13015,L 33,Forslag til lov om ændring af lov om produktio...,Om kompetencebevis.,"Lovforslaget indebærer, at ordningen med, at a...",2014-12-02 13:00:00
3,13004,L 19,Forslag til lov om dansk turisme.,Om dansk turisme.,Lovforslaget etablerer et nationalt turismefor...,2014-12-02 13:00:00
4,12991,L 13,Forslag til lov om ændring af lov om aktiv soc...,Om ændring af formue- og fradragsregler ved ef...,"Formålet med lovforslaget er at sikre, at pers...",2014-12-04 10:00:00


In [8]:
url = "https://oda.ft.dk/api/EmneordSag"

rows = []
next_url = url

while next_url:
    response = requests.get(next_url)
    response.raise_for_status()

    data = response.json()
    rows.extend(data["value"])

    next_url = data.get("odata.nextLink") or data.get("@odata.nextLink")

df_emneord_sag = pd.DataFrame(rows)

print(df_emneord_sag.shape)
df_emneord_sag.head()

(275173, 4)


,id,emneordid,sagid,opdateringsdato
0,749,387,370,2014-08-11T13:23:02.27
1,750,387,371,2014-08-11T13:23:02.52
2,752,226,373,2014-08-11T13:23:04.003
3,8706,2554,574,2014-08-11T19:16:40.833
4,8849,387,4006,2014-08-11T19:19:51.583


In [9]:
df_emneord_sag.columns

Index(['id', 'emneordid', 'sagid', 'opdateringsdato'], dtype='object')

In [10]:
url = "https://oda.ft.dk/api/Emneord"

rows = []
next_url = url

while next_url:
    response = requests.get(next_url)
    response.raise_for_status()

    data = response.json()
    rows.extend(data["value"])

    next_url = data.get("odata.nextLink") or data.get("@odata.nextLink")

df_emneord = pd.DataFrame(rows)

print(df_emneord.shape)
print(df_emneord.columns)

df_emneord.head()

(49227, 4)
Index(['id', 'typeid', 'emneord', 'opdateringsdato'], dtype='object')


,id,typeid,emneord,opdateringsdato
0,1,1,økonomisk redegørelse,2014-12-16T18:44:04.673
1,2,3,Grønland,2014-10-06T09:34:14.377
2,3,4,Retspleje 24,2015-04-21T15:30:57.71
3,4,1,efterretningstjeneste og sikkerhedspolitik,2014-08-11T12:53:19.827
4,5,3,narkotika,2014-08-11T12:53:21.013


In [11]:
df_emneord_full = df_emneord_sag.merge(
    df_emneord,
    left_on="emneordid",
    right_on="id",
    how="left",
    suffixes=("_relation", "_emneord")
)

df_emneord_full.head()

,id_relation,emneordid,sagid,opdateringsdato_relation,id_emneord,typeid,emneord,opdateringsdato_emneord
0,749,387,370,2014-08-11T13:23:02.27,387,1,Tingdok,2014-08-11T12:57:37.433
1,750,387,371,2014-08-11T13:23:02.52,387,1,Tingdok,2014-08-11T12:57:37.433
2,752,226,373,2014-08-11T13:23:04.003,226,1,Statsrevisorerne,2014-08-11T13:23:04.003
3,8706,2554,574,2014-08-11T19:16:40.833,2554,2,miljøudvalget,2014-08-11T19:16:40.833
4,8849,387,4006,2014-08-11T19:19:51.583,387,1,Tingdok,2014-08-11T12:57:37.433


In [15]:
relevant_sagids = set(df_cases["sagid"])

df_emneord_relevant = df_emneord_full[
    df_emneord_full["sagid"].isin(relevant_sagids)
].copy()

In [16]:
print("Relations:", len(df_emneord_relevant))
print(
    "Cases with at least one emneord:",
    df_emneord_relevant["sagid"].nunique()
)
print(
    "Unique emneord:",
    df_emneord_relevant["emneord"].nunique()
)

Relations: 7688
Cases with at least one emneord: 2128
Unique emneord: 884


In [17]:
df_emneord_relevant["typeid"].value_counts().sort_index()

typeid
1    2286
2      39
3    1298
4    4065
Name: count, dtype: int64

In [18]:
url = "https://oda.ft.dk/api/Emneordstype"

data = requests.get(url).json()

df_emneordstype = pd.DataFrame(data["value"])

df_emneordstype

,id,type,opdateringsdato
0,1,Sagsområde,2015-05-27T13:48:24.487
1,2,Ukontrolleret,2015-06-04T08:30:36.683
2,3,Kontrolleret,2015-05-27T13:48:24.487
3,4,Lovsagsnavn,2015-06-08T15:13:27.44
4,5,Thesaurus,2014-08-06T14:43:17.187
5,6,Kontonummer,2014-08-06T14:43:17.187
6,7,Retsgrundlag,2014-08-06T14:43:17.187


In [19]:
type_counts = (
    df_emneord_relevant
    .groupby("typeid")
    .agg(
        relations=("sagid", "size"),
        cases=("sagid", "nunique"),
        unique_emneord=("emneord", "nunique")
    )
    .reset_index()
)

type_counts = type_counts.merge(
    df_emneordstype[["id", "type"]],
    left_on="typeid",
    right_on="id",
    how="left"
)

type_counts[
    ["typeid", "type", "relations", "cases", "unique_emneord"]
]

,typeid,type,relations,cases,unique_emneord
0,1,Sagsområde,2286,2128,254
1,2,Ukontrolleret,39,29,28
2,3,Kontrolleret,1298,555,317
3,4,Lovsagsnavn,4065,1889,289


In [20]:
df_sagsomraade = df_emneord_relevant[
    df_emneord_relevant["typeid"] == 1
].copy()

df_sagsomraade["emneord"].value_counts().head(50)

emneord
strafferet og kriminalitet                        63
COVID-19 pandemien                                60
finansiel virksomhed                              42
finanspolitik                                     42
EU                                                41
retspleje og domstole                             41
folkeskolen                                       41
erhvervsdrivende virksomhed                       40
alternative og vedvarende energikilder            36
energiforsyning                                   33
personskatter                                     33
indvandrere og integration                        32
arbejdsløshedsforsikring og dagpenge              32
indfødsrets og naturalisering                     31
miljøpolitik og miljøbeskyttelse                  29
social service                                    28
boligpolitik                                      28
flygtninge og asylansøgere                        28
dyr og dyrevelfærd                    

In [21]:
print(
    "Cases with Sagsområde:",
    df_sagsomraade["sagid"].nunique(),
    "/",
    df_cases["sagid"].nunique()
)

print(
    "Unique Sagsområder:",
    df_sagsomraade["emneord"].nunique()
)

Cases with Sagsområde: 2128 / 2128
Unique Sagsområder: 254


Sagsområde has 100% coverage of the 2,128 parliamentary cases, and it is clearly intended to describe subject areas. More importantly, the examples look substantively useful: 

*strafferet og kriminalitet, finanspolitik, indvandrere og integration, socialpolitik, energipolitik og energieffektivitet*, etc.

But there's one important complication: 254 Sagsområder is still quite granular, and some appear semantically overlapping:

*arbejdsmarkedspolitik og beskæftigelsesindsats
arbejdsmarkedspolitik og beskæftigelsespolitik
aktiv beskæftigelsesindsats*


So, we have discovered an official controlled starting vocabulary that may allow us to construct broader political topics much more defensibly.

In [22]:
sagsomraader_per_case = (
    df_sagsomraade
    .groupby("sagid")["emneord"]
    .nunique()
)

print(sagsomraader_per_case.value_counts().sort_index())

emneord
1    1985
2     130
3      11
4       2
Name: count, dtype: int64


In [23]:
multi_sagsomraade_ids = sagsomraader_per_case[
    sagsomraader_per_case > 1
].index

df_multi_sagsomraade = (
    df_sagsomraade[
        df_sagsomraade["sagid"].isin(multi_sagsomraade_ids)
    ]
    .groupby("sagid")["emneord"]
    .apply(list)
    .reset_index()
)

df_multi_sagsomraade = df_multi_sagsomraade.merge(
    df_cases[["sagid", "sag_nummer", "sag_titel"]],
    on="sagid",
    how="left"
)

df_multi_sagsomraade[
    ["sag_nummer", "sag_titel", "emneord"]
].head(20)

,sag_nummer,sag_titel,emneord
0,L 9,"Forslag til lov om ændring af momsloven, skatt...","[moms , skattekontrol]"
1,L 10,Forslag til lov om ændring af aktieavancebeska...,"[beskatning af aktier, moms , selskabsskatter ..."
2,L 41,"Forslag til lov om ændring af ligningsloven, f...","[fonds- og foreningsbeskatning, inddrivelse]"
3,L 56,Forslag til lov om ændring af lov om Arbejdsgi...,"[erhvervsuddannelser, uddannelsesstøtte]"
4,L 85,Forslag til lov om ændring af lov om råstoffer...,"[naturbeskyttelse og kystsikring, råstoffer og..."
5,L 103,Forslag til lov om ændring af forældreansvarsl...,"[familieret, udsatte børn og unge]"
6,L 112,Forslag til lov om ændring af lov om jordbruge...,"[landbrug, natur og miljø]"
7,L 130,Forslag til lov om kommunale internationale gr...,"[folkeskolen, frie grundskoler]"
8,L 120,Forslag til lov om ændring af lov om social se...,"[social service, udsatte børn og unge]"
9,L 144,Forslag til lov om ændring af lov om afgift af...,"[energiafgifter , moms ]"


In [24]:
sagsomraader = sorted(
    df_sagsomraade["emneord"].unique()
)

[x for x in sagsomraader if "integr" in x.lower()]

['indvandrere og integration']

In [25]:
[x for x in sagsomraader if "social" in x.lower()]

['social service',
 'sociale forhold',
 'sociale ydelser',
 'socialforvaltning og retssikkerhed',
 'socialpolitik',
 'sundhed og sociale forhold']

In [28]:
print(sagsomraader_per_case.value_counts().sort_index())

df_sagsomraade_vocab = (
    df_sagsomraade
    .groupby("emneord")
    .agg(
        n_cases=("sagid", "nunique")
    )
    .reset_index()
    .sort_values("emneord")
    .reset_index(drop=True)
)

pd.set_option("display.max_rows", 300)

df_sagsomraade_vocab

emneord
1    1985
2     130
3      11
4       2
Name: count, dtype: int64


,emneord,n_cases
0,COVID-19 pandemien,60
1,DSB,4
2,Danida,1
3,Danmarks Statistik,1
4,Det Økonomiske Råd,3
5,EU,41
6,Flygtningenævnet,1
7,NATO,1
8,PET og terrorbekæmpelse,12
9,Statens it,1


## What the 254 Sagsområder actually look like

These aren't simply 254 equally broad "political topics." They range from broad domains such as 

> finanspolitik, 
>
> socialpolitik, 
>
> transportpolitik, and 
>
>miljøpolitik og miljøbeskyttelse 

to much more specific subjects such as 

> børnetilskud,
> 
> tinglysningsafgift,
> 
> værnepligt, and 
>
> indfødsretsprøve.

There are also closely related categories. For example, employment alone includes:

> aktiv beskæftigelsesindsats
>
> arbejdsløshedsforsikring og dagpenge
>
> arbejdsmarkedspolitik og beskæftigelsesindsats
>
>arbejdsmarkedspolitik og beskæftigelsespolitik

## Can an NLP model accurately map candidate-test questions to the Folketing's official subject-area vocabulary?

Candidate-test question

        ↓

Which ODA Sagsområde(s) describe this question?

        ↓

Official Sagsområde

        ↑

Already assigned by Folketinget

        ↑

Parliamentary cases

In [29]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/multilingual-e5-base")

c:\Users\asket\Desktop\Environments\NLP\NLP\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4215.85it/s]


In [30]:
sagsomraader = (
    df_sagsomraade["emneord"]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

len(sagsomraader)

254

In [36]:
sagsomraade_embeddings = model.encode(
    sagsomraader,
    normalize_embeddings=True,
    show_progress_bar=True
)

Batches: 100%|██████████| 8/8 [00:00<00:00, 19.14it/s]


In [37]:
question = "Integrationsydelsen skal hæves"

question_embedding = model.encode(
    [question],
    normalize_embeddings=True
)

In [38]:
import numpy as np

scores = question_embedding[0] @ sagsomraade_embeddings.T

top_indices = np.argsort(scores)[::-1][:10]

for idx in top_indices:
    print(f"{scores[idx]:.4f}  {sagsomraader[idx]}")

0.8945  indvandrere og integration
0.8719  sociale ydelser
0.8643  indfødsrets og naturalisering
0.8639  børne- og ungeydelse
0.8570  arbejdsmarkedsbidrag
0.8544  fonds- og foreningsbeskatning
0.8532  nordisk samarbejde
0.8525  kontanthjælp og uddannelseshjælp
0.8524  værdighed og ældrepolitik
0.8517  arbejdsløshedsforsikring og dagpenge


In [39]:
candidate_test_file = DATA_DIR / "candidate_test" / "Kandidattestdata.xlsx"

df_FV11 = pd.read_excel(candidate_test_file, sheet_name="FV11")
df_FV15 = pd.read_excel(candidate_test_file, sheet_name="FV15")
df_FV19 = pd.read_excel(candidate_test_file, sheet_name="FV19")
df_FV22 = pd.read_excel(candidate_test_file, sheet_name="FV22")

In [40]:
def find_sagsomraader(question, k=5):
    question_embedding = model.encode_query(
        [question],
        normalize_embeddings=True
    )

    scores = question_embedding[0] @ sagsomraade_embeddings.T
    top_indices = np.argsort(scores)[::-1][:k]

    return pd.DataFrame({
        "sagsomraade": [sagsomraader[i] for i in top_indices],
        "score": [scores[i] for i in top_indices]
    })

In [41]:
find_sagsomraader(
    "Integrationsydelsen skal hæves",
    k=5
)

,sagsomraade,score
0,indvandrere og integration,0.894544
1,sociale ydelser,0.871928
2,indfødsrets og naturalisering,0.864276
3,børne- og ungeydelse,0.863947
4,arbejdsmarkedsbidrag,0.856957


In [47]:
df_FV19["Question"].drop_duplicates().sample(
    10,
    random_state=42
).tolist()

['Grænsekontrollen ved den dansk-tyske grænse skal gøres permanent',
 'Der skal øremærkes mere end 14 dage af barselsorloven til fædre',
 'Det skal være obligatorisk for virksomheder at have et bestemt antal kvinder i bestyrelsen',
 'Regionerne skal nedlægges',
 'Den offentlige kulturstøtte skal sænkes',
 'Danmark bruger for mange penge på forsvaret',
 'Aktiv dødshjælp for uhelbredeligt syge skal være lovligt i Danmark',
 'Der skal indføres en særlig klimaafgift på oksekød',
 'Det offentlige tilskud til privatskoler skal sænkes',
 'Man skal automatisk blive organdonor, når man fylder 18 år, medmindre man aktivt framelder sig donorregistret']

In [ ]:
test_questions = [
 'Grænsekontrollen ved den dansk-tyske grænse skal gøres permanent',
 'Der skal øremærkes mere end 14 dage af barselsorloven til fædre',
 'Det skal være obligatorisk for virksomheder at have et bestemt antal kvinder i bestyrelsen',
 'Regionerne skal nedlægges',
 'Den offentlige kulturstøtte skal sænkes',
 'Danmark bruger for mange penge på forsvaret',
 'Aktiv dødshjælp for uhelbredeligt syge skal være lovligt i Danmark',
 'Der skal indføres en særlig klimaafgift på oksekød',
 'Det offentlige tilskud til privatskoler skal sænkes',
 'Man skal automatisk blive organdonor, når man fylder 18 år, medmindre man aktivt framelder sig donorregistret']

for question in test_questions:
    print("\nQUESTION:")
    print(question)

    display(find_sagsomraader(question, k=5))


QUESTION:
Grænsekontrollen ved den dansk-tyske grænse skal gøres permanent


,sagsomraade,score
0,fødevaresikkerhed og fødevarekontrol,0.829149
1,flygtninge og asylansøgere,0.821912
2,skattekontrol,0.819875
3,indfødsretsprøve,0.818973
4,ophold og visum,0.817411



QUESTION:
Der skal øremærkes mere end 14 dage af barselsorloven til fædre


,sagsomraade,score
0,ligestilling og barsel,0.836115
1,færdselsret,0.822666
2,indfødsretsprøve,0.821339
3,afgift af dødsboer og gaver,0.820075
4,patent- og varemærkning,0.816665



QUESTION:
Det skal være obligatorisk for virksomheder at have et bestemt antal kvinder i bestyrelsen


...


QUESTION:
Det offentlige tilskud til privatskoler skal sænkes


,sagsomraade,score
0,ungdoms- og efterskoler,0.848842
1,uddannelsesstøtte,0.845968
2,økonomi og bloktilskud,0.843308
3,tilskud og udligning,0.838223
4,folkeskolen,0.837744



QUESTION:
Man skal automatisk blive organdonor, når man fylder 18 år, medmindre man aktivt framelder sig donorregistret


,sagsomraade,score
0,hjælp til pårørende,0.830119
1,afgift af dødsboer og gaver,0.824840
2,indfødsretsprøve,0.823602
3,miljøafgifter,0.822755
4,videregående uddannelse,0.821774


: 

It is clear that some questions are easier to find topics, especially topics that are actually voted on in parliament. Questions that are about a very narrow or specific topic such as 

> Aktiv dødshjælp for uhelbredeligt syge skal være lovligt i Danmark
>
> or 
>
> Regionerne skal nedlægges

are more difficult to find topics for. The question becomes wether we handle this again as a more broad topic discussion. So we decide that "*Aktiv dødshjælp for uhelbredeligt syge skal være lovligt i Danmark*" is not about that a vote in parliament about that, because that never happened, but more about *patientrettigheder og patientsikkerhed*.